# Fine-tune OLMo 2 1B into a search-relevance judge for an online shop

A shopper types a search. The shop finds many products that match some of the words. Which ones should go on top?
We teach a small model, **AI2's OLMo 2 1B Instruct**, to judge one (search, product) pair with Amazon's four labels,
then use its judgement to **re-rank** the search results. This time the model gets a **classification head**: it
returns four probabilities in one pass instead of writing text. Hugging Face `Trainer`, on the Mac's GPU (MPS).

In [ ]:
import time
from collections import Counter

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
from transformers import (AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, pipeline)

from esci import LABELS, bm25, evaluate, judge_messages, load_esci, pair_text, relevance, row_text, scoreboard, show_ranking

## 0. Check the machine

Same **MacBook Pro** as every lesson. PyTorch reaches the Mac's GPU through **MPS**. Training this one needs about
16 GB of memory.

In [ ]:
!system_profiler SPHardwareDataType | grep -E "Model Name|Chip|Memory"

In [ ]:
torch.backends.mps.is_available()

## 1. Four labels: E, S, C, I

Search for **iphone 15**:

| Product | Label |
|---|---|
| Apple iPhone 15 | **Exact**: what you asked for |
| Apple iPhone 14 | **Substitute**: not it, but you could buy it instead |
| A case for the iPhone 15 | **Complement**: goes with it |
| A blender | **Irrelevant** |

Amazon's people labelled real searches this way: the **Shopping Queries Dataset** (ESCI), on Hugging Face as
[`tasksource/esci`](https://huggingface.co/datasets/tasksource/esci) (Apache-2.0).

## 2. The data

The US shop, Amazon's "hard" set of queries. `load_esci()` downloads the files once (2.5 GB), cleans them and splits
**by query**: a test search and its products are never seen in training. 1,200 searches to train on, 250 to test.

In [ ]:
train, val, test = load_esci()
len(train), len(val), len(test)

In [ ]:
Counter(train.esci_label)

One test search and all its products, with the labels Amazon's annotators gave:

In [ ]:
psp = test[test["query"] == "psp go"]
for _, r in psp.iterrows():
    print(f"{r.esci_label:<11} {r.product_title[:80]}")

What the model reads for one pair: the search, then the product.

In [ ]:
print(row_text(psp.iloc[0]))

## 3. The base model as a judge

OLMo 2 1B Instruct, straight from AI2. We explain the four labels in the system prompt and ask for one word.
`judge()` also keeps the model's probability for each label's first word, so we can rank products by it too.

In [ ]:
MODEL = "allenai/OLMo-2-0425-1B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("mps")
FIRST = [tok.encode(label, add_special_tokens=False)[0] for label in LABELS]

@torch.no_grad()
def judge(text):
    ids = tok.apply_chat_template(judge_messages(text), add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to("mps")
    out = llm.generate(**ids, max_new_tokens=6, do_sample=False, output_scores=True,
                       return_dict_in_generate=True, pad_token_id=tok.pad_token_id)
    answer = tok.decode(out.sequences[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer, out.scores[0][0].float()[FIRST].softmax(-1).tolist()

In [ ]:
IPHONE = ["Apple iPhone 15, 128GB, Black - Unlocked (Renewed)",
          "Apple iPhone 14, 128GB, Midnight - Unlocked (Renewed)",
          "Spigen Ultra Hybrid case designed for iPhone 15",
          "Ninja BL610 Professional 72 Oz Countertop Blender with 1000-Watt Base"]
for title in IPHONE:
    print(f"{judge(pair_text('iphone 15', title))[0]:<11} {title}")

Now all 4,703 test pairs, one at a time.

In [ ]:
answers, base_probs = zip(*[judge(row_text(r)) for _, r in tqdm(test.iterrows(), total=len(test))])
base_preds = [next((l for l in LABELS if l.lower() in a.lower()), "other") for a in answers]
Counter(base_preds)

## 4. Two scores

- **Accuracy**: how many labels are right.
- **NDCG@10**: how good the top 10 order is. Sort each search's products by the model's score and compare with the
  best possible order: 1.0 = perfect, a random order gets about 0.58. The score is one number from the four
  probabilities: an Exact counts 1, a Substitute 0.1, a Complement 0.01, an Irrelevant 0.

In [ ]:
base = evaluate(test, base_preds, [relevance(p) for p in base_probs])
base

For comparison, plain keyword search (BM25), the classic way shops rank results:

In [ ]:
bm25_ndcg = evaluate(test, scores=bm25(test))["ndcg@10"]
bm25_ndcg

## 5. A classification head

Instead of writing a word, the model gets a small new layer on top, the **head**: 2,048 numbers in, 4 scores out, one
per label. One forward pass, four probabilities, no text to parse. `AutoModelForSequenceClassification` builds it;
the head starts random, so it has to learn. LoRA trains small matrices in every layer, and the head trains in full.

In [ ]:
del llm
torch.mps.empty_cache()
torch.manual_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=4, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)},
    dtype=torch.bfloat16)
model.config.pad_token_id = tok.pad_token_id
lora = LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
for p in model.parameters():          # the parts that train in 32-bit, the frozen model stays in 16-bit
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

Each training example is the pair's text and its label number (0 = Exact … 3 = Irrelevant), cut at 128 tokens.

In [ ]:
def encode(df):
    data = Dataset.from_dict({"text": [row_text(r) for _, r in df.iterrows()],
                              "label": [LABELS.index(l) for l in df.esci_label]})
    return data.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True, remove_columns=["text"])

train_ds = encode(train)
train_ds

## 6. Train with the Hugging Face Trainer

One pass over the 23,884 training pairs, 16 at a time.

In [ ]:
args = TrainingArguments(output_dir="checkpoints", num_train_epochs=1, per_device_train_batch_size=16,
                         learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=0.03,
                         logging_steps=50, save_strategy="no", bf16=True, report_to="none", seed=0)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=DataCollatorWithPadding(tok))

In [ ]:
start = time.time()
trainer.train()
train_secs = time.time() - start
gpu_gb = torch.mps.driver_allocated_memory() / 1e9
print(f"{train_secs / 60:.1f} min, GPU memory {gpu_gb:.1f} GB")

## 7. Merge and save

`merge_and_unload()` folds LoRA into the weights: one normal Hugging Face model folder, head included.

In [ ]:
merged = trainer.model.merge_and_unload().to(torch.bfloat16)
merged.save_pretrained("olmo-2-1b-shop-judge")
tok.save_pretrained("olmo-2-1b-shop-judge")

## 8. Use it: one line with `pipeline`

In [ ]:
judge_pipe = pipeline("text-classification", model="olmo-2-1b-shop-judge", device="mps",
                      dtype=torch.bfloat16, top_k=None)

def classify(texts):
    out = judge_pipe(texts, batch_size=32, truncation=True, max_length=128)
    return [[next(x["score"] for x in o if x["label"] == l) for l in LABELS] for o in out]

for title, probs in zip(IPHONE, classify([pair_text("iphone 15", t) for t in IPHONE])):
    best = int(np.argmax(probs))
    print(f"{LABELS[best]:<11} {probs[best]:.2f}  {title}")

## 9. Before vs after

The same 4,703 test pairs.

In [ ]:
start = time.time()
tuned_probs = classify([row_text(r) for _, r in test.iterrows()])
ms = (time.time() - start) / len(test) * 1000
tuned = evaluate(test, [LABELS[int(np.argmax(p))] for p in tuned_probs], [relevance(p) for p in tuned_probs])
tuned, f"{ms:.0f} ms per pair"

In [ ]:
scoreboard(base, tuned, bm25_ndcg, train_secs, gpu_gb, trainer.state.global_step, args.per_device_train_batch_size)

## 10. Re-rank one search

"psp go": its 15 products sorted by the base model's score, then by the fine-tuned judge's.

In [ ]:
rows = psp.index
print("base model"); show_ranking(psp, [relevance(base_probs[i]) for i in rows])
print("fine-tuned"); show_ranking(psp, [relevance(tuned_probs[i]) for i in rows])

## 11. Share it on Hugging Face

The model folder and its model card go to a (private) repo on your account.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
REPO = f"{api.whoami()['name']}/olmo-2-1b-shop-search-judge"
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path="olmo-2-1b-shop-judge", repo_id=REPO, commit_message="OLMo 2 1B + ESCI head, 1 epoch")
api.upload_file(path_or_fileobj="MODEL_CARD.md", path_in_repo="README.md", repo_id=REPO)

## Next: lesson 8

Turn your bank statement into a budget, with Swiss AI's **Apertus 1.5B**, tested on shops it has never seen.